In [7]:
import os
from SAM_finetune.models.sam_model import SAMModel
from SAM_finetune.models.prompt_generator import SAMBoxPromptGenerator, SAMPointPromptGenerator
from SAM_finetune.models.dataset import SAMDataset
from SAM_finetune.utils.config import SAMInferenceConfig, SAMDatasetConfig
import matplotlib.pyplot as plt

/scratch/aimi/am1392/SAM-MedUI/.venv/lib/python3.12/site-packages/albumentations/__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.7'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
/scratch/aimi/am1392/SAM-MedUI/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Load SAM model
sam_config = SAMInferenceConfig(
    device="cpu",
    model_type="vit_b",
    sam_path="../../checkpoints/sam_vit_b_01ec64.pth",
    checkpoint_path="../../runs/run_torchio/best_model.pth",
)

sam_model = SAMModel(config=sam_config)

Load SAM model from  ../../checkpoints/sam_vit_b_01ec64.pth
Loaded checkpoint


In [9]:
dataset_config = SAMDatasetConfig(
    dataset_path="../data/test/",
    remove_nonscar=True,
    sample_size=2,
    point_prompt=True,
    point_prompt_types=['positive'],
    box_prompt=True,
    enable_direction_aug=True,
    enable_size_aug=True,
    number_of_prompts=1,
    image_size=(1024, 1024),
    train=False,
)
dataset = SAMDataset(config=dataset_config)

Loaded 2 images and masks
Removed 1 empty masks
Loaded 1 images and masks


In [ ]:
# Forward pass
image = dataset[0]['image'].unsqueeze(0)

points = {
    'coords': dataset[0]['points_coords'],
    'labels': dataset[0]['points_labels']
}

boxes = dataset[0]['boxes']

masks, iou_prediction = sam_model.forward_one_image(
    image, 
    boxes, 
    points, 
    is_train=False)

# Plot heatmap
heatmap = masks.detach().cpu().numpy().squeeze(0).squeeze(0)

# # Plot heatmap
# plt.imshow(heatmap)
# plt.show()

: 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_sam_predictions(image, sam_model, points=None, boxes=None):
    """
    Create a comparison plot of different SAM prompting strategies
    """
    plt.figure(figsize=(15, 3))
    
    # Setup subplots
    titles = ['Original Image', 'No Prompt', 'Point Prompt', 
             'Box Prompt', 'Combined Prompts']
    
    # Plot original image
    plt.subplot(151)
    plt.imshow(image.squeeze(0).permute(1,2,0))
    plt.title(titles[0])
    plt.axis('off')
    
    # No prompt
    plt.subplot(152)
    masks_no_prompt, _ = sam_model.forward_one_image(image, None, None)
    plt.imshow(masks_no_prompt.detach().cpu().numpy().squeeze())
    plt.title(titles[1])
    plt.axis('off')
    
    # Point prompt
    plt.subplot(153)
    masks_points, _ = sam_model.forward_one_image(image, None, points)
    plt.imshow(masks_points.detach().cpu().numpy().squeeze())
    plt.title(titles[2])
    plt.axis('off')
    
    # Box prompt
    plt.subplot(154)
    masks_boxes, _ = sam_model.forward_one_image(image, boxes, None)
    plt.imshow(masks_boxes.detach().cpu().numpy().squeeze())
    plt.title(titles[3])
    plt.axis('off')
    
    # Combined prompts
    plt.subplot(155)
    masks_combined, _ = sam_model.forward_one_image(image, boxes, points)
    plt.imshow(masks_combined.detach().cpu().numpy().squeeze())
    plt.title(titles[4])
    plt.axis('off')
    
    plt.tight_layout()
    return plt.gcf()

# Usage example
image = dataset[0]['image'].unsqueeze(0)
points = {
    'coords': dataset[0]['points_coords'],
    'labels': dataset[0]['points_labels']
}
boxes = dataset[0]['boxes']

fig = plot_sam_predictions(image, sam_model, points, boxes)
plt.show()

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.3152406..2.3008392].
